# BSTabDiff || Springleaf subset || HDHSS || Mixed || Missing || Tuning || No Permutation

In [ ]:
import os

# Suppress Python warnings before importing third-party packages.
os.environ["PYTHONWARNINGS"] = "ignore"

import warnings
warnings.simplefilter("ignore")
warnings.filterwarnings("ignore")

import gc
import inspect
import time
import random
import contextlib
import sys

import numpy as np
import pandas as pd
import optuna

from sklearn.compose import ColumnTransformer
from sklearn.model_selection import RepeatedStratifiedKFold, StratifiedShuffleSplit
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

from optuna.samplers import TPESampler
from optuna.pruners import MedianPruner

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# PyTorch is required by BSTabDiff and for CUDA resource reporting.
try:
    import torch
except Exception:
    torch = None

# BSTabDiff
from bstabdiff_gobs import FeatureSpec, fit_block_subunit_generator


# ============================================================
# Global runtime configuration
# ============================================================
GLOBAL_SEED = 42

DATASET_NAME = "Springleaf-Subset-Mixed"
CSV_PATH = "Springleaf_subset.csv"
TARGET_COL = "target"
ID_COLS = ["ID"]
EXPECTED_NUM_CLASSES = 2

# Object columns with this timestamp format are converted to continuous
# day offsets while preserving missingness.
SPRINGLEAF_DATETIME_FORMAT = "%d%b%y:%H:%M:%S"
DATETIME_MATCH_THRESHOLD = 0.80

# Very high-cardinality string columns are capped to make both BSTabDiff's
# categorical emission fitting and downstream one-hot Logistic Regression
# computationally manageable. The most frequent K-1 levels are retained;
# all remaining observed levels are grouped into __OTHER__.
MAX_CATEGORIES_PER_FEATURE = 128
OTHER_CATEGORY_TOKEN = "__OTHER__"

# Drop predictors with zero or one observed distinct value. Such columns are
# non-informative and categorical FeatureSpec requires at least two levels.
DROP_CONSTANT_COLUMNS = True

# Fixed to CUDA:6 for BSTabDiff generator training.
REQUESTED_TORCH_DEVICE_STR = "cuda:6"

OPTUNA_TRIALS = 150
OPTUNA_STUDY_NAME = "bstabdiff_Springleaf_mixed_nogobs_cuda6_optuna150"
OPTUNA_STORAGE = None

# Fixed choices requested.
FIXED_PERMUTE_FEATURES = False
FIXED_USE_GOBS = False

# The full dataset is large enough that full Spearman/MI/nearest-neighbor
# fidelity diagnostics can require several GB of RAM. This only limits the
# real-data sample used for the final fidelity block; training and CV still
# use the full dataset.
FIDELITY_MAX_REAL_SAMPLES = 5000

optuna.logging.set_verbosity(optuna.logging.ERROR)


# ============================================================
# Small console helper
# ============================================================
def log(msg):
    print(msg, flush=True)


# ============================================================
# Optuna output suppression + summaries
# ============================================================
@contextlib.contextmanager
def suppress_stdout_stderr():
    with open(os.devnull, "w") as devnull:
        old_stdout, old_stderr = sys.stdout, sys.stderr
        try:
            sys.stdout = devnull
            sys.stderr = devnull
            yield
        finally:
            sys.stdout = old_stdout
            sys.stderr = old_stderr


def _trial_duration_seconds(trial):
    if trial.datetime_start is None or trial.datetime_complete is None:
        return np.nan
    return (trial.datetime_complete - trial.datetime_start).total_seconds()


def build_optuna_trial_dataframe(study):
    rows = []
    best_so_far = -np.inf

    for t in study.trials:
        value = t.value if t.value is not None else np.nan
        if t.state.name == "COMPLETE" and np.isfinite(value):
            best_so_far = max(best_so_far, value)

        row = {
            "trial": t.number,
            "state": t.state.name,
            "value": value,
            "best_so_far": best_so_far if np.isfinite(best_so_far) else np.nan,
            "duration_sec": _trial_duration_seconds(t),
            "datetime_start": str(t.datetime_start) if t.datetime_start is not None else None,
            "datetime_complete": str(t.datetime_complete) if t.datetime_complete is not None else None,
            "error": t.user_attrs.get("error"),
        }

        for k, v in t.params.items():
            row[f"param__{k}"] = v

        rows.append(row)

    df = pd.DataFrame(rows)
    if len(df) == 0:
        return df

    for col in ["value", "best_so_far", "duration_sec"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")
    return df


def get_top_trials_df(trials_df, top_k=10):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    df = df.sort_values(["value", "trial"], ascending=[False, True]).reset_index(drop=True)
    return df.head(top_k)


def get_best_so_far_progression_df(trials_df):
    if trials_df is None or len(trials_df) == 0:
        return pd.DataFrame()

    df = trials_df.copy()
    df = df[df["state"] == "COMPLETE"].copy()
    df = df[np.isfinite(df["value"])].copy()
    if len(df) == 0:
        return pd.DataFrame()

    best_rows = []
    running_best = -np.inf
    for _, row in df.sort_values("trial").iterrows():
        if row["value"] > running_best:
            running_best = row["value"]
            best_rows.append(row)

    if len(best_rows) == 0:
        return pd.DataFrame()

    return pd.DataFrame(best_rows).reset_index(drop=True)


def pretty_param_columns(df):
    cols = [c for c in df.columns if c.startswith("param__")]
    return df.rename(columns={c: c.replace("param__", "") for c in cols})


def print_optuna_summary_tables(study, top_k=10):
    trials_df = build_optuna_trial_dataframe(study)
    top_df = get_top_trials_df(trials_df, top_k=top_k)
    best_prog_df = get_best_so_far_progression_df(trials_df)

    log("\n=== Optuna trial summary ===")
    if len(trials_df) == 0:
        log("No trials found.")
        return trials_df, top_df, best_prog_df

    n_complete = int((trials_df["state"] == "COMPLETE").sum())
    n_pruned = int((trials_df["state"] == "PRUNED").sum())
    n_fail = int((trials_df["state"] == "FAIL").sum())
    n_objective_errors = int(trials_df["error"].notna().sum()) if "error" in trials_df else 0

    log(
        f"Total trials: {len(trials_df)} | COMPLETE: {n_complete} | "
        f"PRUNED: {n_pruned} | FAIL: {n_fail} | objective errors: {n_objective_errors}"
    )

    if len(top_df) > 0:
        top_show = pretty_param_columns(top_df.copy())
        log(f"\n=== Top {min(top_k, len(top_show))} Optuna trials by objective ===")
        log(top_show.to_markdown(index=False))

    if len(best_prog_df) > 0:
        prog_show = pretty_param_columns(best_prog_df.copy())
        log("\n=== Best-so-far progression ===")
        log(prog_show.to_markdown(index=False))

    return trials_df, top_df, best_prog_df


# ============================================================
# Device / seed helpers
# ============================================================
def resolve_torch_device(device_str: str = REQUESTED_TORCH_DEVICE_STR):
    if torch is None:
        return None
    if not torch.cuda.is_available():
        return torch.device("cpu")
    if not device_str.startswith("cuda:"):
        return torch.device(device_str)
    idx = int(device_str.split(":")[1])
    ndev = torch.cuda.device_count()
    if idx >= ndev:
        raise RuntimeError(
            f"Requested {device_str}, but only {ndev} CUDA device(s) are visible."
        )
    return torch.device(device_str)


if torch is not None:
    DEFAULT_TORCH_DEVICE = resolve_torch_device(REQUESTED_TORCH_DEVICE_STR)
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass
else:
    DEFAULT_TORCH_DEVICE = None


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


def cleanup_torch():
    if torch is not None and torch.cuda.is_available():
        try:
            torch.cuda.empty_cache()
        except Exception:
            pass
    gc.collect()


# ============================================================
# 1) Mixed-type Logistic Regression classifier
# ============================================================
def _make_one_hot_encoder():
    # Keep one-hot output sparse: Springleaf contains several categorical
    # columns and dense expansion would consume excessive RAM.
    kwargs = {"handle_unknown": "ignore"}
    params = inspect.signature(OneHotEncoder).parameters
    if "sparse_output" in params:
        kwargs["sparse_output"] = True
    else:
        kwargs["sparse"] = True
    return OneHotEncoder(**kwargs)


def make_clf(continuous_idx, categorical_idx, seed=0):
    continuous_idx = list(map(int, continuous_idx))
    categorical_idx = list(map(int, categorical_idx))

    transformers = []
    if continuous_idx:
        continuous_pipe = Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler(with_mean=False, with_std=True)),
            ]
        )
        transformers.append(("continuous", continuous_pipe, continuous_idx))

    if categorical_idx:
        categorical_pipe = Pipeline(
            [
                ("imp", SimpleImputer(strategy="most_frequent")),
                ("ohe", _make_one_hot_encoder()),
            ]
        )
        transformers.append(("categorical", categorical_pipe, categorical_idx))

    if not transformers:
        raise ValueError("No continuous or categorical feature indices were supplied.")

    preprocessor = ColumnTransformer(
        transformers=transformers,
        remainder="drop",
        # Force a sparse stacked representation whenever categorical
        # one-hot features are present.
        sparse_threshold=1.0,
    )

    return Pipeline(
        [
            ("prep", preprocessor),
            (
                "clf",
                LogisticRegression(
                    max_iter=5000,
                    class_weight="balanced",
                    solver="saga",
                    n_jobs=-1,
                    random_state=seed,
                ),
            ),
        ]
    )


# ============================================================
# 2) Classification metric helpers
# ============================================================
def _predict_proba_binary(clf, X):
    if hasattr(clf, "predict_proba"):
        proba = np.asarray(clf.predict_proba(X), dtype=float)
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba.ravel()

    if hasattr(clf, "decision_function"):
        scores = np.asarray(clf.decision_function(X), dtype=float)
        return 1.0 / (1.0 + np.exp(-scores))

    raise ValueError("Classifier has neither predict_proba nor decision_function.")


def _safe_binary_auc_or_acc(clf, X_val, y_val):
    y_val = np.asarray(y_val).astype(int)

    if len(np.unique(y_val)) < 2:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))

    try:
        proba = _predict_proba_binary(clf, X_val)
        return float(roc_auc_score(y_val, proba))
    except Exception:
        pred = clf.predict(X_val)
        return float(accuracy_score(y_val, pred))


def _sanitize_mixed_matrix(X):
    X = np.asarray(X, dtype=np.float32).copy()
    X[np.isposinf(X) | np.isneginf(X)] = np.nan
    return X


# ============================================================
# 3) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    continuous_idx,
    categorical_idx,
    generator_fit_fn=None,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
):
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided.")

    X_real = _sanitize_mixed_matrix(X_real)
    y_real = np.asarray(y_real).astype(int)

    classes, counts = np.unique(y_real, return_counts=True)
    if len(classes) != 2:
        raise ValueError(
            f"Binary classification requires exactly 2 classes, but found "
            f"{len(classes)}: {classes.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "RepeatedStratifiedKFold with n_splits=5 requires at least "
            f"5 samples per class. Class counts: "
            f"{dict(zip(classes.tolist(), counts.tolist()))}"
        )

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)
        X_syn_fold = _sanitize_mixed_matrix(X_syn_fold)
        y_syn_fold = np.asarray(y_syn_fold).astype(int)

        if len(np.unique(y_syn_fold)) < 2:
            raise RuntimeError(
                f"Fold {fold_id}: the synthetic dataset contains fewer than two classes."
            )

        # TSTR: train on synthetic, test on real.
        clf_tstr = make_clf(continuous_idx, categorical_idx, seed=fold_seed)
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(float(roc_auc_score(yte, proba_t)))
        tstr_acc.append(float(accuracy_score(yte, pred_t)))

        # TRTR: train on real, test on real.
        if do_trtr:
            clf_trtr = make_clf(continuous_idx, categorical_idx, seed=fold_seed)
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(float(roc_auc_score(yte, proba_r)))
            trtr_acc.append(float(accuracy_score(yte, pred_r)))

        # C2ST: distinguish real=1 from synthetic=0.
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf(continuous_idx, categorical_idx, seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(float(roc_auc_score(ymix_te, proba_m)))
            c2st_acc.append(float(accuracy_score(ymix_te, pred_m)))

        log(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, "
            f"AUC={tstr_auc[-1]:.4f}"
        )
        cleanup_torch()

    def _mean_std(values):
        values = np.asarray(values, dtype=float)
        return float(np.mean(values)), float(np.std(values))

    tstr_auc_mean, tstr_auc_std = _mean_std(tstr_auc)
    tstr_acc_mean, tstr_acc_std = _mean_std(tstr_acc)

    out = {
        "tstr_auc_mean": tstr_auc_mean,
        "tstr_auc_std": tstr_auc_std,
        "tstr_acc_mean": tstr_acc_mean,
        "tstr_acc_std": tstr_acc_std,
        "n_folds": len(tstr_auc),
    }

    if do_trtr and len(trtr_auc) > 0:
        trtr_auc_mean, trtr_auc_std = _mean_std(trtr_auc)
        trtr_acc_mean, trtr_acc_std = _mean_std(trtr_acc)
        out.update(
            {
                "trtr_auc_mean": trtr_auc_mean,
                "trtr_auc_std": trtr_auc_std,
                "trtr_acc_mean": trtr_acc_mean,
                "trtr_acc_std": trtr_acc_std,
            }
        )

    if do_c2st and len(c2st_auc) > 0:
        c2st_auc_mean, c2st_auc_std = _mean_std(c2st_auc)
        c2st_acc_mean, c2st_acc_std = _mean_std(c2st_acc)
        out.update(
            {
                "c2st_auc_mean": c2st_auc_mean,
                "c2st_auc_std": c2st_auc_std,
                "c2st_acc_mean": c2st_acc_mean,
                "c2st_acc_std": c2st_acc_std,
            }
        )

    return out


# ============================================================
# 4) CSV saving helpers
# ============================================================
def _decode_categorical_column(values, decoding):
    values = np.asarray(values, dtype=float)
    out = np.full(values.shape, np.nan, dtype=object)
    valid = np.isfinite(values)
    if np.any(valid):
        codes = np.rint(values[valid]).astype(int)
        decoded = [decoding.get(int(code), np.nan) for code in codes]
        out[valid] = decoded
    return out


def save_synth_csv(
    X_syn,
    y_syn=None,
    out_csv="Springleaf_mixed_synth_BSTabDiff_n200_nogobs_cuda6.csv",
    feature_names=None,
    categorical_idx=None,
    category_decoders=None,
    target_class_names=None,
    target_col=TARGET_COL,
):
    X_syn = _sanitize_mixed_matrix(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    if len(feature_names) != m:
        raise ValueError("feature_names length does not match synthetic feature count.")

    df = pd.DataFrame(X_syn, columns=feature_names)

    categorical_idx = [] if categorical_idx is None else list(categorical_idx)
    category_decoders = {} if category_decoders is None else category_decoders
    for j in categorical_idx:
        name = feature_names[int(j)]
        decoder = category_decoders.get(int(j), {})
        df[name] = _decode_categorical_column(X_syn[:, int(j)], decoder)

    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        if target_class_names is not None:
            target_class_names = np.asarray(target_class_names, dtype=object)
            decoded_target = np.asarray(
                [target_class_names[int(v)] for v in y_syn], dtype=object
            )
            df.insert(0, target_col, decoded_target)
        else:
            df.insert(0, target_col, y_syn)

    df.to_csv(out_csv, index=False)
    log(
        f"Saved CSV: {out_csv} | shape={df.shape} | "
        f"nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(
    R_syn,
    out_csv="Springleaf_mixed_synth_BSTabDiff_n200_mask_nogobs_cuda6.csv",
):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    log(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | "
        f"observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 5) BSTabDiff Optuna tuning
# ============================================================
def _sample_bstabdiff_params(trial):
    params = {}

    params["M"] = trial.suggest_categorical("M", [8, 12, 16, 24, 32, 48, 64])
    params["prior_type"] = trial.suggest_categorical("prior_type", ["diffusion", "flow"])
    params["prior_epochs"] = trial.suggest_categorical("prior_epochs", [300, 500, 800, 1200, 1600])
    params["prior_batch"] = trial.suggest_categorical("prior_batch", [16, 32, 64, 128])
    params["prior_lr"] = trial.suggest_float("prior_lr", 5e-5, 5e-3, log=True)
    params["use_ema"] = True
    params["ema_decay"] = trial.suggest_float("ema_decay", 0.99, 0.9999, log=True)

    # Fixed: no GO-BS, no permutation.
    params["use_gobs"] = FIXED_USE_GOBS
    params["permute_features"] = FIXED_PERMUTE_FEATURES

    # Placeholders for compatibility.
    params["gobs_num_clusters"] = 7
    params["gobs_metric"] = "correlation"
    params["gobs_bins"] = 32
    params["gobs_top_k"] = None
    params["gobs_refine_order"] = False
    params["gobs_direction_select"] = False
    params["gobs_refine_passes"] = 0
    params["gobs_boundary_refine_passes"] = 1
    params["gobs_boundary_window"] = 8
    params["gobs_lambda_cross"] = 1.0
    params["gobs_gamma_balance"] = 1e-6

    params["n_syn_inner"] = trial.suggest_categorical("n_syn_inner", [64, 100, 128, 200, 256])

    return params


class _FrozenTrialAdapter:
    def __init__(self, params_dict):
        self.params_dict = params_dict

    def suggest_categorical(self, name, choices):
        if name in self.params_dict:
            return self.params_dict[name]
        return choices[0]

    def suggest_int(self, name, low, high, step=1, log=False):
        if name in self.params_dict:
            return int(self.params_dict[name])
        return low

    def suggest_float(self, name, low, high, step=None, log=False):
        if name in self.params_dict:
            return float(self.params_dict[name])
        return low


def _fit_bstabdiff_with_params(
    Xtr,
    ytr,
    params,
    feature_specs,
    device=REQUESTED_TORCH_DEVICE_STR,
    seed=0,
    verbose_every=0,
):
    Xtr = _sanitize_mixed_matrix(Xtr)
    ytr = np.asarray(ytr).astype(int)

    if len(feature_specs) != Xtr.shape[1]:
        raise ValueError(
            f"feature_specs has {len(feature_specs)} entries but X has {Xtr.shape[1]} features."
        )

    gen = fit_block_subunit_generator(
        X=Xtr,
        feature_specs=feature_specs,
        y=ytr,
        M=params["M"],
        blocks=None,
        permute_features=False,
        prior_type=params["prior_type"],
        device=device,
        seed=seed,
        prior_epochs=params["prior_epochs"],
        prior_batch=params["prior_batch"],
        prior_lr=params["prior_lr"],
        verbose_every=verbose_every,
        save_dir=None,
        save_name="bstabdiff_Springleaf_mixed_nogobs_cuda6",
        save_best=True,
        use_ema=params["use_ema"],
        ema_decay=params["ema_decay"],
        return_train_info=False,
        use_gobs=False,
        gobs_num_clusters=params["gobs_num_clusters"],
        gobs_metric=params["gobs_metric"],
        gobs_bins=params["gobs_bins"],
        gobs_top_k=params["gobs_top_k"],
        gobs_refine_order=params["gobs_refine_order"],
        gobs_direction_select=params["gobs_direction_select"],
        gobs_refine_passes=params["gobs_refine_passes"],
        gobs_boundary_refine_passes=params["gobs_boundary_refine_passes"],
        gobs_boundary_window=params["gobs_boundary_window"],
        gobs_lambda_cross=params["gobs_lambda_cross"],
        gobs_gamma_balance=params["gobs_gamma_balance"],
        gobs_use_cpu_kmeans=False,
    )
    return gen


def tune_bstabdiff_once(
    X,
    y,
    feature_specs,
    continuous_idx,
    categorical_idx,
    device=REQUESTED_TORCH_DEVICE_STR,
    n_trials=OPTUNA_TRIALS,
    seed=GLOBAL_SEED,
    inner_val_size=0.2,
    show_progress_bar=True,
):
    X = _sanitize_mixed_matrix(X)
    y = np.asarray(y).astype(int)

    splitter = StratifiedShuffleSplit(
        n_splits=1,
        test_size=inner_val_size,
        random_state=seed,
    )
    (itr, iva), = splitter.split(X, y)
    X_inner_tr, y_inner_tr = X[itr], y[itr]
    X_inner_va, y_inner_va = X[iva], y[iva]

    def objective(trial):
        params = _sample_bstabdiff_params(trial)
        trial_seed = int(seed + 1000 + trial.number)

        try:
            gen_trial = _fit_bstabdiff_with_params(
                Xtr=X_inner_tr,
                ytr=y_inner_tr,
                params=params,
                feature_specs=feature_specs,
                device=device,
                seed=trial_seed,
                verbose_every=0,
            )

            X_syn, R_syn, y_syn = gen_trial.sample(n=int(params["n_syn_inner"]))
            X_syn = _sanitize_mixed_matrix(X_syn)
            y_syn = np.asarray(y_syn).astype(int)

            if len(np.unique(y_syn)) < 2:
                return 0.0

            proxy_clf = make_clf(continuous_idx, categorical_idx, seed=trial_seed)
            proxy_clf.fit(X_syn, y_syn)
            score = _safe_binary_auc_or_acc(
                proxy_clf,
                X_inner_va,
                y_inner_va,
            )

            score_adj = float(score) - 1e-4 * float(params["M"])
            return score_adj

        except Exception as exc:
            trial.set_user_attr("error", f"{type(exc).__name__}: {exc}"[:1000])
            return 0.0

        finally:
            cleanup_torch()

    sampler = TPESampler(
        seed=seed,
        multivariate=True,
        warn_independent_sampling=False,
    )
    pruner = MedianPruner(n_startup_trials=10, n_warmup_steps=0)

    study = optuna.create_study(
        direction="maximize",
        sampler=sampler,
        pruner=pruner,
        study_name=OPTUNA_STUDY_NAME,
        storage=OPTUNA_STORAGE,
        load_if_exists=False,
    )

    with suppress_stdout_stderr():
        study.optimize(objective, n_trials=n_trials, show_progress_bar=show_progress_bar)

    trials_df = build_optuna_trial_dataframe(study)
    successful = trials_df[
        (trials_df["state"] == "COMPLETE")
        & trials_df["error"].isna()
        & np.isfinite(trials_df["value"])
    ]
    if len(successful) == 0:
        example_errors = trials_df["error"].dropna().head(5).tolist()
        raise RuntimeError(
            "Every Optuna trial failed before producing a valid objective. "
            f"Example errors: {example_errors}"
        )

    best_full = _sample_bstabdiff_params(_FrozenTrialAdapter(dict(study.best_trial.params)))
    best_full["permute_features"] = False
    best_full["use_gobs"] = False

    trials_df, top_trials_df, best_progression_df = print_optuna_summary_tables(study, top_k=10)
    return best_full, study, trials_df, top_trials_df, best_progression_df


def make_generator_fit_fn_from_best_params(
    best_params,
    feature_specs,
    device=REQUESTED_TORCH_DEVICE_STR,
    verbose_every=0,
):
    def _fit(Xtr, ytr, seed=0):
        gen = _fit_bstabdiff_with_params(
            Xtr=_sanitize_mixed_matrix(Xtr),
            ytr=np.asarray(ytr).astype(int),
            params=best_params,
            feature_specs=feature_specs,
            device=device,
            seed=seed,
            verbose_every=verbose_every,
        )
        return gen
    return _fit


# ============================================================
# 6) Mixed-type fidelity metric helpers
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    x = x[np.isfinite(x)]
    y = y[np.isfinite(y)]
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return np.nan, np.nan

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _fill_with_reference_medians(X, reference):
    X = np.asarray(X, dtype=float).copy()
    reference = np.asarray(reference, dtype=float)
    medians = np.nanmedian(reference, axis=0)
    medians[~np.isfinite(medians)] = 0.0
    bad = ~np.isfinite(X)
    if np.any(bad):
        X[bad] = np.broadcast_to(medians, X.shape)[bad]
    return X


def _rank_transform(X):
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col, kind="mergesort")
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def _categorical_tv_distance(x_real, x_syn, n_categories):
    x_real = np.asarray(x_real, dtype=float)
    x_syn = np.asarray(x_syn, dtype=float)

    # Missing is an additional state at index n_categories.
    states_real = np.full(len(x_real), n_categories, dtype=int)
    states_syn = np.full(len(x_syn), n_categories, dtype=int)

    obs_r = np.isfinite(x_real)
    obs_s = np.isfinite(x_syn)
    states_real[obs_r] = np.clip(np.rint(x_real[obs_r]).astype(int), 0, n_categories - 1)
    states_syn[obs_s] = np.clip(np.rint(x_syn[obs_s]).astype(int), 0, n_categories - 1)

    p = np.bincount(states_real, minlength=n_categories + 1).astype(float)
    q = np.bincount(states_syn, minlength=n_categories + 1).astype(float)
    p /= max(p.sum(), 1.0)
    q /= max(q.sum(), 1.0)
    return float(0.5 * np.abs(p - q).sum()), p, q


def _prepare_mi_matrix(X, continuous_idx, categorical_idx, category_sizes, reference):
    X = np.asarray(X, dtype=float).copy()
    reference = np.asarray(reference, dtype=float)

    if continuous_idx:
        cont = np.asarray(continuous_idx, dtype=int)
        medians = np.nanmedian(reference[:, cont], axis=0)
        medians[~np.isfinite(medians)] = 0.0
        block = X[:, cont]
        bad = ~np.isfinite(block)
        if np.any(bad):
            block[bad] = np.broadcast_to(medians, block.shape)[bad]
        X[:, cont] = block

    for j in categorical_idx:
        j = int(j)
        K = int(category_sizes[j])
        col = X[:, j]
        missing = ~np.isfinite(col)
        col[~missing] = np.clip(np.rint(col[~missing]), 0, K - 1)
        col[missing] = K
        X[:, j] = col

    return X


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names,
    continuous_idx,
    categorical_idx,
    category_sizes,
    generator=None,
    random_state=0,
):
    X_real = _sanitize_mixed_matrix(X_real).astype(float)
    X_syn = _sanitize_mixed_matrix(X_syn).astype(float)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."

    missing_real = np.mean(~np.isfinite(X_real), axis=0)
    missing_syn = np.mean(~np.isfinite(X_syn), axis=0)
    missing_abs_diff = np.abs(missing_real - missing_syn)

    continuous_idx = np.asarray(continuous_idx, dtype=int)
    categorical_idx = np.asarray(categorical_idx, dtype=int)

    ks_stats = np.full(X_real.shape[1], np.nan, dtype=float)
    w1_stats = np.full(X_real.shape[1], np.nan, dtype=float)
    for j in continuous_idx:
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    categorical_metrics = {}
    for j in categorical_idx:
        K = int(category_sizes[int(j)])
        tv, p_real, p_syn = _categorical_tv_distance(
            X_real[:, j], X_syn[:, j], n_categories=K
        )
        categorical_metrics[feature_names[int(j)]] = {
            "total_variation": tv,
            "real_probabilities_including_missing": p_real,
            "synthetic_probabilities_including_missing": p_syn,
            "missing_rate_real": float(missing_real[j]),
            "missing_rate_syn": float(missing_syn[j]),
            "missing_rate_abs_diff": float(missing_abs_diff[j]),
        }

    pearson_mean = pearson_max = np.nan
    spearman_mean = spearman_max = np.nan
    higher = {
        "mean_abs_diff": np.nan,
        "var_abs_diff": np.nan,
        "skew_abs_diff": np.nan,
        "kurt_abs_diff": np.nan,
    }

    if len(continuous_idx) >= 2:
        real_cont = _fill_with_reference_medians(
            X_real[:, continuous_idx], X_real[:, continuous_idx]
        )
        syn_cont = _fill_with_reference_medians(
            X_syn[:, continuous_idx], X_real[:, continuous_idx]
        )

        pearson_real = np.corrcoef(real_cont, rowvar=False)
        pearson_syn = np.corrcoef(syn_cont, rowvar=False)
        mask_offdiag = ~np.eye(len(continuous_idx), dtype=bool)
        pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
        pearson_abs_diff = pearson_abs_diff[np.isfinite(pearson_abs_diff)]
        if len(pearson_abs_diff):
            pearson_mean = float(np.mean(pearson_abs_diff))
            pearson_max = float(np.max(pearson_abs_diff))

        ranks_real = _rank_transform(real_cont)
        ranks_syn = _rank_transform(syn_cont)
        spearman_real = np.corrcoef(ranks_real, rowvar=False)
        spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
        spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
        spearman_abs_diff = spearman_abs_diff[np.isfinite(spearman_abs_diff)]
        if len(spearman_abs_diff):
            spearman_mean = float(np.mean(spearman_abs_diff))
            spearman_max = float(np.max(spearman_abs_diff))

        mean_r, var_r, skew_r, kurt_r = _moments_1d(real_cont)
        mean_s, var_s, skew_s, kurt_s = _moments_1d(syn_cont)
        higher = {
            "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
            "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
            "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
            "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
        }

    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        real_mi_X = _prepare_mi_matrix(
            X_real,
            continuous_idx.tolist(),
            categorical_idx.tolist(),
            category_sizes,
            X_real,
        )
        syn_mi_X = _prepare_mi_matrix(
            X_syn,
            continuous_idx.tolist(),
            categorical_idx.tolist(),
            category_sizes,
            X_real,
        )
        discrete_mask = np.zeros(X_real.shape[1], dtype=bool)
        discrete_mask[categorical_idx] = True
        mi_real = mutual_info_classif(
            real_mi_X,
            y_real,
            discrete_features=discrete_mask,
            random_state=random_state,
        )
        mi_syn = mutual_info_classif(
            syn_mi_X,
            y_syn,
            discrete_features=discrete_mask,
            random_state=random_state,
        )
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    nn_stats = {}
    try:
        Xr_nn = _prepare_mi_matrix(
            X_real,
            continuous_idx.tolist(),
            categorical_idx.tolist(),
            category_sizes,
            X_real,
        )
        Xs_nn = _prepare_mi_matrix(
            X_syn,
            continuous_idx.tolist(),
            categorical_idx.tolist(),
            category_sizes,
            X_real,
        )
        mu = Xr_nn.mean(axis=0)
        sd = Xr_nn.std(axis=0)
        sd[sd < 1e-8] = 1.0
        Xr_nn = (Xr_nn - mu) / sd
        Xs_nn = (Xs_nn - mu) / sd

        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(Xr_nn)
        dists, idx = nn.kneighbors(Xs_nn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "categorical_metrics": categorical_metrics,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "missing_rate_real_per_feature": missing_real,
        "missing_rate_syn_per_feature": missing_syn,
        "missing_rate_abs_diff_per_feature": missing_abs_diff,
        "missing_rate_abs_diff_mean": float(np.mean(missing_abs_diff)),
        "missing_rate_abs_diff_max": float(np.max(missing_abs_diff)),
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": float(np.mean(~np.isfinite(X_real))),
        "nan_frac_syn": float(np.mean(~np.isfinite(X_syn))),
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 7) Load and encode the Springleaf mixed-type dataset
# ============================================================
def _looks_like_springleaf_datetime(series, threshold=DATETIME_MATCH_THRESHOLD):
    values = series.dropna().astype(str).str.strip()
    if len(values) == 0:
        return False
    pattern = r"^\d{2}[A-Za-z]{3}\d{2}:\d{2}:\d{2}:\d{2}$"
    return float(values.str.match(pattern).mean()) >= float(threshold)


def _datetime_to_days(series):
    raw = series.astype("string").str.strip()
    parsed = pd.to_datetime(
        raw,
        format=SPRINGLEAF_DATETIME_FORMAT,
        errors="coerce",
    )
    days = (parsed - pd.Timestamp("1970-01-01")) / pd.Timedelta(days=1)
    return days.astype(float)


def _encode_categorical_series(
    series,
    max_categories=MAX_CATEGORIES_PER_FEATURE,
    other_token=OTHER_CATEGORY_TOKEN,
):
    raw = series.astype("string").str.strip()
    raw = raw.mask(raw.eq(""), pd.NA)

    counts = raw.value_counts(dropna=True)
    original_n_categories = int(len(counts))

    if original_n_categories < 2:
        raise ValueError(
            f"Categorical column '{series.name}' has fewer than two observed levels."
        )

    collapsed = False
    kept_levels = counts.index.astype(str).tolist()
    if original_n_categories > int(max_categories):
        # Reserve one level for all less-frequent observed categories.
        kept_levels = counts.index[: int(max_categories) - 1].astype(str).tolist()
        keep_set = set(kept_levels)
        raw = raw.map(
            lambda value: (
                pd.NA
                if pd.isna(value)
                else (str(value) if str(value) in keep_set else other_token)
            )
        ).astype("string")
        kept_levels.append(other_token)
        collapsed = True

    # Use deterministic codes based on the retained-level order.
    encoding = {level: idx for idx, level in enumerate(kept_levels)}
    decoding = {idx: level for level, idx in encoding.items()}
    encoded = raw.map(encoding).astype("Float64").astype(float)

    return encoded, encoding, decoding, {
        "original_n_categories": original_n_categories,
        "encoded_n_categories": int(len(encoding)),
        "collapsed_to_other": bool(collapsed),
        "missing_count": int(encoded.isna().sum()),
        "top_original_counts": counts.head(10).to_dict(),
    }


def _categorize_regime(n, m):
    rho = float(m / n)
    if m > 1000 and n < 1000 and rho > 2:
        regime = "HDLSS"
    elif m > 1000 and n > 10_000 and 0.005 < rho <= 2:
        regime = "HDHSS"
    elif m <= 100 and n > 10_000 and rho <= 0.01:
        regime = "LDHSS"
    elif m <= 100 and n <= 1000 and rho <= 0.05:
        regime = "LDLSS"
    else:
        regime = "MixedRegime"
    return regime, rho


def load_springleaf_mixed_dataset(
    csv_path=CSV_PATH,
    target_col=TARGET_COL,
    id_cols=ID_COLS,
    expected_num_classes=EXPECTED_NUM_CLASSES,
):
    df = pd.read_csv(csv_path, low_memory=False)

    if target_col not in df.columns:
        raise ValueError(
            f"Target column '{target_col}' not found. "
            f"Final columns: {list(df.columns[-15:])}"
        )
    if df[target_col].isna().any():
        raise ValueError(f"Target column '{target_col}' contains missing values.")

    # Encode binary target to contiguous 0/1 IDs.
    y_raw = df[target_col].to_numpy()
    if pd.api.types.is_numeric_dtype(df[target_col]):
        unique_target = np.sort(pd.unique(y_raw))
        target_encoding = {value: idx for idx, value in enumerate(unique_target)}
        y = np.asarray([target_encoding[value] for value in y_raw], dtype=int)
        class_names = np.asarray(unique_target, dtype=object)
    else:
        y, class_names = pd.factorize(
            df[target_col].astype("string").str.strip(),
            sort=True,
        )
        y = np.asarray(y, dtype=int)
        class_names = np.asarray(class_names, dtype=object)

    classes, counts = np.unique(y, return_counts=True)
    if expected_num_classes is not None and len(classes) != expected_num_classes:
        raise ValueError(
            f"Expected {expected_num_classes} target classes, but found "
            f"{len(classes)}: {class_names.tolist()}"
        )
    if counts.min() < 5:
        raise ValueError(
            "The requested 5-fold stratified CV requires at least 5 samples "
            f"per class. Counts: {dict(zip(classes.tolist(), counts.tolist()))}"
        )

    drop_ids = [c for c in id_cols if c in df.columns]
    X_df = df.drop(columns=[target_col] + drop_ids).copy()

    # Remove all-missing and constant predictors before type inference.
    nunique = X_df.nunique(dropna=True)
    dropped_constant_cols = nunique[nunique <= 1].index.tolist()
    if DROP_CONSTANT_COLUMNS and dropped_constant_cols:
        X_df = X_df.drop(columns=dropped_constant_cols)

    datetime_cols = []
    categorical_cols = []
    categorical_reports = {}
    category_encoders_by_name = {}
    category_decoders_by_name = {}

    # Process all object/string columns. Timestamp-like columns are encoded as
    # continuous days; remaining strings are genuine categorical variables.
    object_cols = list(X_df.select_dtypes(include=["object", "string", "category"]).columns)
    for col in object_cols:
        if _looks_like_springleaf_datetime(X_df[col]):
            X_df[col] = _datetime_to_days(X_df[col])
            datetime_cols.append(col)
        else:
            encoded, enc, dec, report = _encode_categorical_series(X_df[col])
            X_df[col] = encoded
            categorical_cols.append(col)
            categorical_reports[col] = report
            category_encoders_by_name[col] = enc
            category_decoders_by_name[col] = dec

    # Any remaining predictors must be numeric after conversion.
    nonnumeric = [
        c for c in X_df.columns
        if not pd.api.types.is_numeric_dtype(X_df[c])
    ]
    if nonnumeric:
        raise ValueError(
            f"Predictors remain nonnumeric after preprocessing: {nonnumeric[:20]}"
        )

    feature_names = list(X_df.columns)
    categorical_idx = [feature_names.index(c) for c in categorical_cols]
    categorical_idx_set = set(categorical_idx)
    continuous_idx = [j for j in range(len(feature_names)) if j not in categorical_idx_set]

    # Preserve NaNs. BSTabDiff constructs the observation mask with np.isfinite.
    X = X_df.to_numpy(dtype=np.float32)
    X[np.isposinf(X) | np.isneginf(X)] = np.nan

    feature_specs = []
    category_sizes = {}
    category_decoders = {}
    for j, name in enumerate(feature_names):
        if j in categorical_idx_set:
            K = len(category_encoders_by_name[name])
            if K < 2:
                raise ValueError(
                    f"Categorical feature '{name}' has invalid K={K}."
                )
            feature_specs.append(
                FeatureSpec(name=name, kind="categorical", n_categories=K)
            )
            category_sizes[j] = K
            category_decoders[j] = dict(category_decoders_by_name[name])
        else:
            feature_specs.append(FeatureSpec(name=name, kind="continuous"))

    n, m = X.shape
    regime, rho = _categorize_regime(n=n, m=m)

    return {
        "X": X,
        "y": y,
        "feature_names": feature_names,
        "class_names": class_names,
        "feature_specs": feature_specs,
        "continuous_idx": continuous_idx,
        "categorical_idx": categorical_idx,
        "category_sizes": category_sizes,
        "category_decoders": category_decoders,
        "categorical_reports": categorical_reports,
        "datetime_cols": datetime_cols,
        "dropped_id_cols": drop_ids,
        "dropped_constant_cols": dropped_constant_cols,
        "raw_shape": tuple(df.shape),
        "regime": regime,
        "rho": rho,
    }


# ============================================================
# 8) Main
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    log(f"Using torch device: {DEFAULT_TORCH_DEVICE}")
    log(f"Fixed permute_features: {FIXED_PERMUTE_FEATURES}")
    log(f"Fixed use_gobs: {FIXED_USE_GOBS}")
    log("Classifier: sparse mixed-type Logistic Regression only")

    data = load_springleaf_mixed_dataset(
        csv_path=CSV_PATH,
        target_col=TARGET_COL,
        id_cols=ID_COLS,
        expected_num_classes=EXPECTED_NUM_CLASSES,
    )

    X = data["X"]
    y = data["y"]
    feature_names = data["feature_names"]
    original_class_names = data["class_names"]
    feature_specs = data["feature_specs"]
    continuous_idx = data["continuous_idx"]
    categorical_idx = data["categorical_idx"]
    category_sizes = data["category_sizes"]
    category_decoders = data["category_decoders"]

    class_counts = dict(zip(*np.unique(y, return_counts=True)))
    log(
        f"Loaded {DATASET_NAME}: raw_shape={data['raw_shape']}, "
        f"model_X.shape={X.shape}, y.shape={y.shape}, "
        f"encoded_classes={np.unique(y).tolist()}, class_counts={class_counts}"
    )
    log(f"Original target values: {original_class_names.tolist()}")
    log(f"Dropped ID columns: {data['dropped_id_cols']}")
    log(f"Dropped constant/all-missing predictors: {len(data['dropped_constant_cols'])}")
    log(
        f"Predictors: {len(continuous_idx)} continuous + "
        f"{len(categorical_idx)} categorical"
    )
    log(f"Datetime-like object columns converted to continuous days: {len(data['datetime_cols'])}")
    log(
        f"Dataset regime: {data['regime']} | n={X.shape[0]} | "
        f"m={X.shape[1]} | rho=m/n={data['rho']:.6f}"
    )
    log(f"Overall predictor missing fraction: {np.mean(~np.isfinite(X)):.6g}")

    log("\nCategorical feature summary:")
    for col, report in data["categorical_reports"].items():
        log(
            f"  {col}: original_K={report['original_n_categories']}, "
            f"encoded_K={report['encoded_n_categories']}, "
            f"collapsed={report['collapsed_to_other']}, "
            f"missing={report['missing_count']}"
        )

    # ============================================================
    # (0) ONE-TIME Optuna tuning on the full Springleaf dataset
    # ============================================================
    log(
        f"\n=== Running ONE-TIME Optuna tuning for BSTabDiff "
        f"({OPTUNA_TRIALS} trials, no GO-BS, no permutation, cuda:6) "
        f"on full {DATASET_NAME} ==="
    )
    t_tune0 = time.time()

    best_bstabdiff_params, study, trials_df, top_trials_df, best_progression_df = tune_bstabdiff_once(
        X=X,
        y=y,
        feature_specs=feature_specs,
        continuous_idx=continuous_idx,
        categorical_idx=categorical_idx,
        device=REQUESTED_TORCH_DEVICE_STR,
        n_trials=OPTUNA_TRIALS,
        seed=GLOBAL_SEED,
        inner_val_size=0.2,
        show_progress_bar=True,
    )

    t_tune = time.time() - t_tune0

    log("\n=== Best BSTabDiff params from single Optuna study ===")
    log(str(best_bstabdiff_params))
    log(f"Best Optuna objective value: {study.best_value:.6f}")
    log(f"Tuning time: {t_tune:.2f} sec")

    trials_csv = "bstabdiff_Springleaf_mixed_optuna_trials_full_nogobs_cuda6.csv"
    top_trials_csv = "bstabdiff_Springleaf_mixed_optuna_top_trials_nogobs_cuda6.csv"
    progression_csv = "bstabdiff_Springleaf_mixed_optuna_best_progression_nogobs_cuda6.csv"

    trials_df.to_csv(trials_csv, index=False)
    top_trials_df.to_csv(top_trials_csv, index=False)
    best_progression_df.to_csv(progression_csv, index=False)

    log("\nSaved Optuna summaries:")
    log(f"  - {trials_csv}")
    log(f"  - {top_trials_csv}")
    log(f"  - {progression_csv}")

    generator_fit_fn = make_generator_fit_fn_from_best_params(
        best_params=best_bstabdiff_params,
        feature_specs=feature_specs,
        device=REQUESTED_TORCH_DEVICE_STR,
        verbose_every=0,
    )

    # ============================================================
    # (A) Fit BSTabDiff once on full data for reporting
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        try:
            torch.cuda.reset_peak_memory_stats(device=DEFAULT_TORCH_DEVICE)
        except Exception:
            pass

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)

    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated(device=DEFAULT_TORCH_DEVICE)
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    log(f"\n=== BSTabDiff generator: resource usage (trained once on full {DATASET_NAME}) ===")
    log(f"  Train time: {gen_train_time:.2f} sec")
    log(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    log(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    N_SYN = 200
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)
    X_syn_global = _sanitize_mixed_matrix(X_syn_global)
    y_syn_global = np.asarray(y_syn_global).astype(int)

    synth_csv = "Springleaf_mixed_synth_BSTabDiff_n200_nogobs_cuda6.csv"
    mask_csv = "Springleaf_mixed_synth_BSTabDiff_n200_mask_nogobs_cuda6.csv"

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv=synth_csv,
        feature_names=feature_names,
        categorical_idx=categorical_idx,
        category_decoders=category_decoders,
        target_class_names=original_class_names,
        target_col=TARGET_COL,
    )
    save_synth_mask_csv(R_syn_global, out_csv=mask_csv)

    # ============================================================
    # (B) Strict fold-wise 5x5 TSTR/TRTR/C2ST
    # ============================================================
    log(
        "\n=== Evaluating STRICT BSTabDiff synthetic + mixed-type "
        "Logistic Regression (5x5 TSTR/TRTR/C2ST) ==="
    )

    res = eval_5x5_tstr_foldwise_gen(
        X_real=X,
        y_real=y,
        continuous_idx=continuous_idx,
        categorical_idx=categorical_idx,
        generator_fit_fn=generator_fit_fn,
        n_syn=N_SYN,
        do_trtr=True,
        do_c2st=True,
        seed=0,
    )

    log(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
    log(f"  TSTR AUC: {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
    log(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
    log(f"  TRTR AUC: {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
    log(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
    log(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

    summary_df = pd.DataFrame(
        [{
            "Classifier": "Sparse mixed-type Logistic Regression",
            "TSTR ACC (mean ± std)": f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}",
            "TSTR AUC (mean ± std)": f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}",
            "TRTR ACC (mean ± std)": f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}",
            "TRTR AUC (mean ± std)": f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}",
            "C2ST ACC (mean ± std)": f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}",
            "C2ST AUC (mean ± std)": f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}",
        }]
    )

    summary_csv = "bstabdiff_Springleaf_mixed_cv_summary_lr_nogobs_cuda6.csv"
    summary_df.to_csv(summary_csv, index=False)
    log(f"\n=== {DATASET_NAME}: 5x5 TSTR/TRTR/C2ST summary ===")
    log(summary_df.to_markdown(index=False))
    log(f"Saved CV summary: {summary_csv}")

    # ============================================================
    # (C) Mixed-type fidelity diagnostics
    # ============================================================
    if len(X) > FIDELITY_MAX_REAL_SAMPLES:
        fidelity_rng = np.random.RandomState(GLOBAL_SEED)
        fidelity_idx = fidelity_rng.choice(
            len(X), size=FIDELITY_MAX_REAL_SAMPLES, replace=False
        )
        X_fidelity_real = X[fidelity_idx]
        y_fidelity_real = y[fidelity_idx]
        log(
            f"\nFidelity diagnostics use a reproducible real-data subsample of "
            f"{FIDELITY_MAX_REAL_SAMPLES}/{len(X)} rows for memory control."
        )
    else:
        X_fidelity_real = X
        y_fidelity_real = y

    fidelity = compute_fidelity_metrics(
        X_real=X_fidelity_real,
        y_real=y_fidelity_real,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        continuous_idx=continuous_idx,
        categorical_idx=categorical_idx,
        category_sizes=category_sizes,
        generator=gen_full,
        random_state=0,
    )

    log(f"\n=== {DATASET_NAME}: Fidelity metrics (real vs synthetic) ===")
    cont_ks = fidelity["ks_per_feature"][np.asarray(continuous_idx, dtype=int)]
    cont_w1 = fidelity["wasserstein_per_feature"][np.asarray(continuous_idx, dtype=int)]
    log(
        f"  Continuous KS: mean={np.nanmean(cont_ks):.4f}, "
        f"max={np.nanmax(cont_ks):.4f}"
    )
    log(
        f"  Continuous Wasserstein-1: mean={np.nanmean(cont_w1):.4f}, "
        f"max={np.nanmax(cont_w1):.4f}"
    )
    log(
        f"  Pearson |corr_real-corr_syn|: "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    log(
        f"  Spearman |corr_real-corr_syn|: "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    log("  Continuous higher-order moments (mean absolute difference):")
    log(f"    mean: {higher['mean_abs_diff']:.4e}")
    log(f"    var:  {higher['var_abs_diff']:.4e}")
    log(f"    skew: {higher['skew_abs_diff']:.4e}")
    log(f"    kurt: {higher['kurt_abs_diff']:.4e}")

    if fidelity["categorical_metrics"]:
        tv_values = [
            v["total_variation"]
            for v in fidelity["categorical_metrics"].values()
            if np.isfinite(v["total_variation"])
        ]
        log(
            f"  Categorical total-variation distance: "
            f"mean={np.mean(tv_values):.4f}, max={np.max(tv_values):.4f}"
        )

    log(
        f"  Missing-rate absolute difference: "
        f"mean={fidelity['missing_rate_abs_diff_mean']:.4f}, "
        f"max={fidelity['missing_rate_abs_diff_max']:.4f}"
    )

    if "mi_abs_diff_mean" in fidelity:
        log(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    log(
        f"  NaN fraction real={fidelity['nan_frac_real']:.4e}, "
        f"synthetic={fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        log("  Privacy (synthetic to nearest real distance):")
        log(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Using torch device: cuda:6
Fixed permute_features: False
Fixed use_gobs: False
Classifier: sparse mixed-type Logistic Regression only
Loaded Springleaf-Subset-Mixed: raw_shape=(29046, 1934), model_X.shape=(29046, 1876), y.shape=(29046,), encoded_classes=[0, 1], class_counts={np.int64(0): np.int64(22291), np.int64(1): np.int64(6755)}
Original target values: [0, 1]
Dropped ID columns: ['ID']
Dropped constant/all-missing predictors: 56
Predictors: 1854 continuous + 22 categorical
Datetime-like object columns converted to continuous days: 16
Dataset regime: HDHSS | n=29046 | m=1876 | rho=m/n=0.064587
Overall predictor missing fraction: 0.0112387

Categorical feature summary:
  VAR_0001: original_K=3, encoded_K=3, collapsed=False, missing=0
  VAR_0005: original_K=4, encoded_K=4, collapsed=False, missing=0
  VAR_0200: original_K=6074, encoded_K=128, collapsed=True, missing=11
  VAR_0214: original_K=2, encoded_K=2, collapsed=False, missing=29044
  VAR_0226: original_K=2, encoded_K=2, collapse

  0%|          | 0/150 [00:00<?, ?it/s]


=== Optuna trial summary ===
Total trials: 150 | COMPLETE: 150 | PRUNED: 0 | FAIL: 0 | objective errors: 0

=== Top 10 Optuna trials by objective ===
|   trial | state    |    value |   best_so_far |   duration_sec | datetime_start             | datetime_complete          | error   |   M | prior_type   |   prior_epochs |   prior_batch |    prior_lr |   ema_decay |   n_syn_inner |
|--------:|:---------|---------:|--------------:|---------------:|:---------------------------|:---------------------------|:--------|----:|:-------------|---------------:|--------------:|------------:|------------:|--------------:|
|     106 | COMPLETE | 0.66203  |      0.66203  |        75.2986 | 2026-07-27 05:51:26.281740 | 2026-07-27 05:52:41.580308 |         |   8 | flow         |           1600 |            32 | 0.000212861 |    0.993617 |           256 |
|      25 | COMPLETE | 0.660515 |      0.660515 |        70.5525 | 2026-07-27 04:26:15.854405 | 2026-07-27 04:27:26.406884 |         |   8 | flow     